# Participant Starter - Test your ISU System (Track A)

This notebook gives an overview of the provided data, including **simulated** scenes, **real** counterparts, and channels.

It provide an example pipeline for the translation and writing of the final submission folder `submissions/<your_team_name>/` in the expected format.

The task is to implement the image **translation technique**: given a simulated image
(and its scene label), produce a generated image that looks real while keeping the same objects and geometry as the simulated source.

See `README_CONTESTANTS.md` for the exact submission rules (file naming, allowed formats,
what you may and may not change).

In [2]:
import sys
from pathlib import Path

VENV_PYTHON = Path(sys.executable).resolve()
if Path(sys.prefix).resolve() == Path(sys.base_prefix).resolve():
    raise RuntimeError(
        'Select the Track A virtual-environment kernel (.venv or venv) and restart the notebook.'
    )

print('Using environment:', VENV_PYTHON)

Using environment: /usr/bin/python3.11


In [ ]:
import json
import platform
import shutil
import time

from huggingface_hub import snapshot_download
from PIL import Image

# Change this to your team name. This becomes the folder name under submissions/.
TEAM_NAME = 'your_team_name'
DATASET_ID = 'ISU-Test/isu-challenge-dataset'

TRACK_A_DIR = Path.cwd() if Path.cwd().name == 'track_a' else Path.cwd() / 'track_a'
REPO_ROOT = TRACK_A_DIR.parent
DATA_DIR = REPO_ROOT / 'data'
SUBMISSION_DIR = TRACK_A_DIR / 'submissions' / TEAM_NAME

# Download the dataset outside track_a when it is not already available locally.
if not any(DATA_DIR.rglob('*_sim.png')):
    print(f'Downloading {DATASET_ID} to {DATA_DIR}...')
    snapshot_download(
        repo_id=DATASET_ID,
        repo_type='dataset',
        local_dir=str(DATA_DIR),
    )

print(f'Using data directory: {DATA_DIR}')

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Using data directory: /home/ubuntu/testing/ISU-Challenge/Starterkit-ICSE-2027/data


## 1. Load the provided data

The notebook loads data from the repository-level `data/` folder, which is outside `track_a/`. If the folder does not contain simulated scenes, the setup cell downloads `ISU-Test/isu-challenge-dataset` from Hugging Face automatically.

The provided data are organized by scene stem. Files with the same stem belong to the same synthetic scene:

- `images/<stem>_sim.png` — a **simulated** RGB source render that you must translate (one output is required for each simulated image).
- `labels/<stem>_label.json` — categorical scene labels which should not chage after transformation.
- `canny/<stem>_canny.png` — a Canny edge map derived from the simulated RGB image.
- `instance_seg/<stem>_instance_seg.png` — a false-color instance-segmentation map. Use `instance_seg/instance_seg_legend.json` to interpret its colors.
- `depth/png/<stem>_sim_depth.png` and `depth/exr/<stem>_sim_depth.exr` — depth maps in PNG and EXR formats when available.
- Real target-domain reference images — real in-car photos used to describe the target visual distribution.

The simulated RGB image, label, and auxiliary channels describe the same camera view and can be used together by your translation method. Real reference images are not pixel-aligned crops of the simulated images, do not need to match them one-to-one, and may have different resolutions. The generated RGB output must preserve the source scene's objects, geometry, framing, and viewpoint.

In [ ]:
def discover_scenes(data_dir):
    scenes = {}
    image_dir = data_dir / 'images'
    label_dir = data_dir / 'labels'

    for sim_path in sorted(image_dir.glob('*_sim.png')):
        stem = sim_path.stem[:-len('_sim')]
        label_path = label_dir / f'{stem}_label.json'
        if label_path.exists():
            scenes[stem] = {
                'simulated': sim_path,
                'label': label_path,
            }
    return scenes


def discover_reference_images(data_dir):
    return sorted(
        path for path in data_dir.rglob('*')
        if path.is_file() and path.suffix.lower() in {'.jpg', '.jpeg'}
    )


scenes = discover_scenes(DATA_DIR)
reference_images = discover_reference_images(DATA_DIR)

print(f'Found {len(scenes)} simulated scenes: {list(scenes)}')
print(f'Found {len(reference_images)} real reference images.')

Found 1 simulated scenes: ['sample_0001']
Found 0 real reference images.


## Browse scenes and auxiliary channels

Use the controls below to move through the discovered scenes. Each scene displays its simulated RGB image, Canny edges, instance-segmentation map, depth image when a PNG depth file is available, and label JSON. The auxiliary files are matched using the scene stem.

In [ ]:
import ipywidgets as widgets
from IPython.display import HTML, clear_output, display


def auxiliary_paths(stem):
    return {
        'Canny': DATA_DIR / 'canny' / f'{stem}_canny.png',
        'Instance segmentation': DATA_DIR / 'instance_seg' / f'{stem}_instance_seg.png',
        'Depth': DATA_DIR / 'depth' / 'png' / f'{stem}_sim_depth.png',
    }


def image_widget(path, width=260):
    if not path.exists():
        return widgets.HTML(f'<i>Not available</i><br><small>{path}</small>')
    return widgets.Image(value=path.read_bytes(), format=path.suffix.lstrip('.'), width=width)


def show_scene(stem):
    scene = scenes[stem]
    label = json.loads(scene['label'].read_text())
    panels = [
        widgets.VBox([widgets.HTML('<b>Simulated RGB</b>'), image_widget(scene['simulated'])]),
    ]
    for title, path in auxiliary_paths(stem).items():
        panels.append(widgets.VBox([widgets.HTML(f'<b>{title}</b>'), image_widget(path)]))

    clear_output(wait=True)
    display(widgets.HBox(panels, layout=widgets.Layout(overflow='auto')))
    display(HTML(f'<b>Scene:</b> {stem}'))
    display(HTML(f'<pre>{json.dumps(label, indent=2, ensure_ascii=True)}</pre>'))


if scenes:
    scene_names = list(scenes)
    scene_index = {'value': 0}
    selector = widgets.Dropdown(options=scene_names, value=scene_names[0], description='Scene:')
    previous = widgets.Button(description='Previous', icon='arrow-left')
    next_button = widgets.Button(description='Next', icon='arrow-right')
    output = widgets.Output()

    def render_selected(change=None):
        with output:
            show_scene(selector.value)

    def move_scene(step):
        scene_index['value'] = (scene_index['value'] + step) % len(scene_names)
        selector.value = scene_names[scene_index['value']]

    def on_selector_change(change):
        if change['name'] == 'value' and change['new'] in scene_names:
            scene_index['value'] = scene_names.index(change['new'])
            render_selected()

    selector.observe(on_selector_change)
    previous.on_click(lambda _: move_scene(-1))
    next_button.on_click(lambda _: move_scene(1))
    display(widgets.HBox([previous, next_button, selector]))
    display(output)
    render_selected()
else:
    display(HTML('<b>No scenes found.</b> Run the setup and discovery cells after the dataset has been downloaded.'))

In [ ]:
# Sanity check: look at one scene before writing any code.
if not scenes:
    print('No simulated scenes found. Run the setup and discovery cells after the dataset has been downloaded.')
else:
    example_stem = next(iter(scenes))
    example = scenes[example_stem]
    sim_image = Image.open(example['simulated']).convert('RGB')
    label = json.loads(example['label'].read_text())

    if reference_images:
        real_image = Image.open(reference_images[0]).convert('RGB')

        # Side-by-side preview, simulated then one real reference, scaled to the same height.
        height = 360
        sim_preview = sim_image.resize((int(sim_image.width * height / sim_image.height), height))
        real_preview = real_image.resize((int(real_image.width * height / real_image.height), height))
        preview = Image.new('RGB', (sim_preview.width + real_preview.width + 10, height), 'white')
        preview.paste(sim_preview, (0, 0))
        preview.paste(real_preview, (sim_preview.width + 10, 0))
        print(f'{example_stem}: simulated {sim_image.size} (left) vs real reference {real_image.size} (right)')
        display(preview)
    else:
        print(f'{example_stem}: simulated {sim_image.size}; no real reference images found')

    display(label)

StopIteration: 

## 2. Your translation technique

Implement `translate` below. It receives:

- one simulated image 
- its label dict (optional, to condition on it)
- **kwargs (optional channel information, e.g., seg maps, canny edges, or depht maps) 

and must return one PIL image, that:

- keeps the same width, height, framing, and camera viewpoint as the simulated input
  (resizing is allowed but reduces fidelity/semantic scores — see `README_CONTESTANTS.md`)
- does not crop, rotate, mirror, or change scene geometry
- does not add, remove, or move scene objects
- is RGB

The placeholder below just returns the simulated image unchanged, so the rest of this notebook
runs end to end out of the box and produces a (trivial/bad) baseline submission. Replace the
body with your own model / pipeline.

In [ ]:
def translate(simulated_image: Image.Image, stem: str, label: dict, **kwargs) -> Image.Image:
    """Translate one simulated scene into a real-looking one."""
    # >>> REPLACE THIS WITH YOUR TECHNIQUE <<<
    # This placeholder is the identity function: it returns the simulated image
    # unchanged, which will score poorly on realism but perfectly on semantic
    # preservation. Use it to confirm the pipeline runs before you plug in a model.
    return simulated_image.convert('RGB')

## 3. Write the submission folder

This builds `submissions/<team_name>/` with the four subfolders the evaluator expects: `simulated/` and `reference/` are copies of the provided data, `labels/` is a copy of the scene labels, and `generated/` holds your `translate()` output for every scene. The cell also measures translation time and writes `generation_times.json` beside these folders. Do not hand-edit `simulated/` or `reference/` after this runs.

In [ ]:
for name in ('simulated', 'generated', 'reference', 'labels'):
    (SUBMISSION_DIR / name).mkdir(parents=True, exist_ok=True)

# Copy the real images as a separate reference distribution; they do not pair one-to-one with scenes.
for reference_path in reference_images:
    shutil.copy(reference_path, SUBMISSION_DIR / 'reference' / reference_path.name)

total_generation_seconds = 0.0
for index, (stem, paths) in enumerate(scenes.items(), 1):
    print(f'{index}/{len(scenes)}: {stem}')
    label = json.loads(paths['label'].read_text())
    simulated_image = Image.open(paths['simulated']).convert('RGB')

    started = time.perf_counter()
    generated_image = translate(simulated_image, stem, label)
    total_generation_seconds += time.perf_counter() - started
    generated_image.save(SUBMISSION_DIR / 'generated' / f'{stem}.png')

    shutil.copy(paths['simulated'], SUBMISSION_DIR / 'simulated' / paths['simulated'].name)
    shutil.copy(paths['label'], SUBMISSION_DIR / 'labels' / paths['label'].name)

generation_times = {
    'total_seconds': total_generation_seconds,
    'scenes': len(scenes),
    'hardware': {
        'platform': platform.platform(),
        'processor': platform.processor(),
        'python': platform.python_version(),
    },
}
(SUBMISSION_DIR / 'generation_times.json').write_text(
    json.dumps(generation_times, indent=2, ensure_ascii=True)
)
print(f'\nWrote submission to {SUBMISSION_DIR}')
print(f'Total translation time: {total_generation_seconds:.3f} seconds')

## 4. Next steps

Run `track_a_challenge_evaluator.ipynb` to score `submissions/<your_team_name>/`. Re-run this notebook (with your real `translate()` implementation) any time you want to regenerate your submission before scoring.